# BERT Model Experiments

This notebook fine-tunes a pretrained BERT model to classify crisis-related social media posts into humanitarian need categories. The model's performance will be evaluated using the project's training, validation, and test datasets.

In [24]:
#imports
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from pathlib import Path

In [25]:
#load datasets
DATA_DIR = Path(
    "/Users/emilybartell/Documents/CS1684 Crisis NLP/cs1684-crisis-nlp"
) / "data" / "raw" / "event_aware_en"

train_df = pd.read_csv(
    DATA_DIR / "crisis_consolidated_humanitarian_filtered_lang_en_w_event_info_train.tsv",
    sep="\t"
)

dev_df = pd.read_csv(
    DATA_DIR / "crisis_consolidated_humanitarian_filtered_lang_en_w_event_info_dev.tsv",
    sep="\t"
)

test_df = pd.read_csv(
    DATA_DIR / "crisis_consolidated_humanitarian_filtered_lang_en_w_event_info_test.tsv",
    sep="\t"
)

In [26]:
#inspect datasets
print("Training shape:", train_df.shape)
print("Validation shape:", dev_df.shape)
print("Test shape:", test_df.shape)

print("\nTraining label counts:")
print(train_df["class_label"].value_counts())

print("\nMissing values in training data:")
print(train_df[["text", "class_label", "event"]].isnull().sum())

Training shape: (61235, 7)
Validation shape: (8957, 7)
Test shape: (17365, 7)

Training label counts:
class_label
not_humanitarian                       36109
donation_and_volunteering               5184
requests_or_needs                       4840
sympathy_and_support                    3549
infrastructure_and_utilities_damage     3541
affected_individual                     2454
caution_and_advice                      2101
injured_or_dead_people                  1945
response_efforts                         780
missing_and_found_people                 373
displaced_and_evacuations                359
Name: count, dtype: int64

Missing values in training data:
text           0
class_label    0
event          0
dtype: int64


## Initialize BERT

Load the pretrained BERT tokenizer and classification model. The classifier is configured for the project's 11 categories and uses Apple GPU acceleration when available.

In [27]:
#initialize the model
MODEL_NAME = "bert-base-uncased"
NUM_LABELS = 11

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS
)

device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cpu"
)

model.to(device)

print("Model initialized successfully.")
print("Device:", device)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model initialized successfully.
Device: mps


### Prepare Text and Labels

BERT requires text to be converted into tokens and labels to be represented numerically. This section creates a consistent mapping for the 11 humanitarian categories and tokenizes the training, validation, and test datasets.

In [28]:
# Create a consistent mapping of category names to numeric labels
label_names = sorted(train_df["class_label"].unique())

label2id = {label: i for i, label in enumerate(label_names)}
id2label = {i: label for label, i in label2id.items()}

# Convert category names to numeric labels
train_df["label"] = train_df["class_label"].map(label2id)
dev_df["label"] = dev_df["class_label"].map(label2id)
test_df["label"] = test_df["class_label"].map(label2id)

print("Number of classes:", len(label_names))
print("\nLabel mapping:")
print(label2id)

Number of classes: 11

Label mapping:
{'affected_individual': 0, 'caution_and_advice': 1, 'displaced_and_evacuations': 2, 'donation_and_volunteering': 3, 'infrastructure_and_utilities_damage': 4, 'injured_or_dead_people': 5, 'missing_and_found_people': 6, 'not_humanitarian': 7, 'requests_or_needs': 8, 'response_efforts': 9, 'sympathy_and_support': 10}


In [29]:
# convert each post into token IDs BERT can process
MAX_LENGTH = 128

train_encodings = tokenizer(
    train_df["text"].astype(str).tolist(),
    truncation=True,
    padding="max_length",
    max_length=MAX_LENGTH
)

dev_encodings = tokenizer(
    dev_df["text"].astype(str).tolist(),
    truncation=True,
    padding="max_length",
    max_length=MAX_LENGTH
)

test_encodings = tokenizer(
    test_df["text"].astype(str).tolist(),
    truncation=True,
    padding="max_length",
    max_length=MAX_LENGTH
)

print("Tokenization complete.")
print("Training examples:", len(train_encodings["input_ids"]))
print("Validation examples:", len(dev_encodings["input_ids"]))
print("Test examples:", len(test_encodings["input_ids"]))

Tokenization complete.
Training examples: 61235
Validation examples: 8957
Test examples: 17365


In [30]:
#check results
print("Example text:")
print(train_df["text"].iloc[0])

print("\nEncoded token IDs:")
print(train_encodings["input_ids"][0][:20])

print("\nNumeric label:", train_df["label"].iloc[0])
print("Original category:", train_df["class_label"].iloc[0])

Example text:
None Approximately 100km long firebreaks have been constructed by Indonesia and elsewhere.

Encoded token IDs:
[101, 3904, 3155, 2531, 22287, 2146, 2543, 23890, 2015, 2031, 2042, 3833, 2011, 6239, 1998, 6974, 1012, 102, 0, 0]

Numeric label: 4
Original category: infrastructure_and_utilities_damage


In [31]:
# 1. Import the functions from our Python file
from src.bert_model import load_data, prepare_labels, tokenize_data

# 2. Load all three datasets
train_df, dev_df, test_df = load_data()

# 3. Convert category names into numeric labels
train_df, dev_df, test_df, label2id, id2label = prepare_labels(
    train_df, dev_df, test_df
)

# 4. Tokenize the text and prepare PyTorch datasets
train_dataset, dev_dataset, test_dataset = tokenize_data(
    train_df, dev_df, test_df, tokenizer
)

# 5. Display a summary of the results
print("All tests passed!")
print(f"Training examples: {len(train_dataset):,}")
print(f"Validation examples: {len(dev_dataset):,}")
print(f"Test examples: {len(test_dataset):,}")
print(f"Number of categories: {len(label2id)}")
print(f"Device available: {device}")

# 6. Check one example
example = train_dataset[0]
print("\nExample input shape:", example["input_ids"].shape)
print("Example label:", example["labels"].item())

All tests passed!
Training examples: 61,235
Validation examples: 8,957
Test examples: 17,365
Number of categories: 11
Device available: mps

Example input shape: torch.Size([128])
Example label: 4
